In [10]:
import os

import pandas as pd
import networkx as nx

from pyvis.network import Network
from ipywidgets import Dropdown, Button, HBox, VBox, Output, Label, IntSlider
from IPython.display import display, HTML, IFrame

In [8]:
# --- Load node annotations ---
data_path = "../../flatten/processed-data/20251023-s288c-annotated-PPI-net.pkl"
df = pd.read_pickle(data_path)

# Make sure node IDs are strings (helps avoid mismatch with graph nodes)
df["node"] = df["node"].astype(str)

print("Annotation DataFrame shape:", df.shape)
print("Example columns:", df.columns.tolist()[:10])

# --- Load edge list ---
edge_path = "../../0-download-inputs/data-files/The_Yeast_Interactome_edges.csv"
edges = pd.read_csv(edge_path)

print("Edge file columns:", edges.columns.tolist())

# --- Choose edge columns ---
# If you know the column names for the interactors, replace these two lines
# e.g., u_col, v_col = "Interactor_A", "Interactor_B"
#edge_cols = list(edges.columns)
#if len(edge_cols) < 2:
#    raise ValueError("Edge list must contain at least two columns representing endpoints.")

u_col, v_col = "source", "target"  # <-- adjust if needed

# Ensure node IDs are strings
edges[u_col] = edges[u_col].astype(str)
edges[v_col] = edges[v_col].astype(str)

# --- Build graph ---
G = nx.from_pandas_edgelist(edges, u_col, v_col)

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Optional: intersect nodes with annotations
annot_nodes = set(df["node"])
graph_nodes = set(G.nodes())
overlap = annot_nodes & graph_nodes
print(f"Nodes with annotations & in graph: {len(overlap)}")

# We'll just keep the full graph, but only annotated nodes will show rich tooltips.

Annotation DataFrame shape: (3927, 200)
Example columns: ['node', '_wkshell', 'degree_centrality', 'betweenness_centrality', 'eigenvector_centrality', 'closeness_centrality', 'load_centrality', 'pagerank', 'information_centrality', 'CentralityCosDist_rank']
Edge file columns: ['Binary (literature evidence APID)', 'cor_val_>=4_[σ]', 'cor_val_>=5_[σ]', 'cor_val_[σ]', 'Count of publications', 'experimentral System', 'F_FDR_0.001', 'F_FDR_0.01', 'F_FDR_0.05', 'F_FDR_1e-04', 'Highlight Novel Stylel In Cytoscape (shows score if published)', 'Inter-cluster edge', 'interaction', 'name', 'Pubmed Ids', 'R_FDR_0.001', 'R_FDR_0.01', 'R_FDR_0.05', 'R_FDR_1e-04', 'score_cor', 'score_FDR', 'score_FDR+cor', 'selected', 'shared interaction', 'shared name', 'source', 'Source Gene names (SGD/UniProt-primary or ordered locus)', 'target', 'Target Gene names  (SGD/UniProt-primary or ordered locus)']
Graph has 3927 nodes and 31004 edges
Nodes with annotations & in graph: 3927


In [19]:
list(df.columns)

['node',
 '_wkshell',
 'degree_centrality',
 'betweenness_centrality',
 'eigenvector_centrality',
 'closeness_centrality',
 'load_centrality',
 'pagerank',
 'information_centrality',
 'CentralityCosDist_rank',
 'CentralityCosDist_similarity_score',
 'has_verified_sequence',
 'sequence',
 'UniProtKB-AC',
 'DeepTMHMM_trimmed_sequence',
 'DeepTMHMM_class',
 'cleavage_site_start',
 'signalP_trimmed_sequence',
 'ProteinName',
 'GO_terms',
 'GO_terms_human_readable',
 'localization_keywords',
 'parsed_functions',
 'parsed_PTMs',
 'ptm_bronze',
 'ptm_gold',
 'ptm_silver',
 'IDR_count',
 'IDR_sequences',
 'IDR_ranges',
 'N_aa_disordered',
 'disorder_fraction',
 'is_disordered_0.05',
 'is_disordered_0.10',
 'is_disordered_0.15',
 'is_disordered_0.20',
 'is_disordered_0.25',
 'is_disordered_0.30',
 'is_disordered_0.35',
 'is_disordered_0.40',
 'is_disordered_0.45',
 'is_disordered_0.50',
 'is_disordered_0.55',
 'is_disordered_0.60',
 'is_disordered_0.65',
 'is_disordered_0.70',
 'is_disordered_0

In [20]:
df["interacting_chaperones"]

0              []
1       [YBR072W]
2       [YBR072W]
3              []
4              []
          ...    
3922           []
3923           []
3924           []
3925           []
3926           []
Name: interacting_chaperones, Length: 3927, dtype: object

In [17]:
# Pre-index annotation DataFrame for quick lookup
annot = df.set_index("node")

# Valid nodes that exist in both df and graph
valid_nodes = sorted(annot.index.intersection(G.nodes()))

gene_dropdown = Dropdown(
    options=valid_nodes,
    description="Gene:",
    layout={"width": "300px"}
)

max_neighbors_slider = IntSlider(
    value=30,
    min=5,
    max=200,
    step=5,
    description="Max neighbors:",
    continuous_update=False,
    layout={"width": "300px"}
)

update_button = Button(
    description="Show neighbors (pyvis)",
    tooltip="Display the selected node and its immediate neighbors (interactive)",
    button_style=""
)

out_plot = Output()
out_table = Output()


def show_neighborhood_pyvis(gene_id: str, max_neighbors: int | None = None):
    """Build the 1-hop neighborhood of gene_id and display it with pyvis."""
    out_plot.clear_output(wait=True)
    out_table.clear_output(wait=True)

    if gene_id not in G:
        with out_plot:
            print(f"{gene_id} is not present in the NetworkX graph.")
        return

    # --- Build 1-hop neighborhood ---
    neighbors = list(G.neighbors(gene_id))

    if max_neighbors is not None and len(neighbors) > max_neighbors:
        neighbors = sorted(neighbors, key=lambda n: G.degree(n), reverse=True)[:max_neighbors]

    sub_nodes = [gene_id] + neighbors
    subG = G.subgraph(sub_nodes).copy()

    # --- Create pyvis network ---
    net = Network(
        notebook=True,
        height="600px",
        width="100%",
        bgcolor="white",
        font_color="black",
    )
    net.barnes_hut(
    gravity=-2000,          # was gravitationalConstant
    central_gravity=0.7,    # was centralGravity
    spring_length=70,       # was springLength
    spring_strength=0.04,   # was springStrength
    damping=0.85,
    overlap=0.1             # was avoidOverlap
    )

    for n in subG.nodes():
        label = n

        # Default tooltip = just node name
        title = str(n)

        if n in annot.index:
            row = annot.loc[n]

            # disorder_fraction -> f_disordered
            if "disorder_fraction" in row.index:
                val = row["disorder_fraction"]
                try:
                    val_str = f"{float(val):.3f}"
                except Exception:
                    val_str = str(val)
                # Plain text tooltip with newline
                title = f"{n}\n Fraction disordered: {val_str}"

        # Color & size
        if n == gene_id:
            color = "#0077BB"  # selected node color
            size = 25
        else:
            color = "lightgray"
            size = 15

        net.add_node(
            n,
            label=label,
            title=title,  # plain text with \n
            color=color,
            size=size,
        )

    # Add edges
    for u, v in subG.edges():
        net.add_edge(u, v)

    # Save & display as an IFrame
    html_file = "ppi_neighborhood.html"
    net.show(html_file)

    with out_plot:
        display(IFrame(src=html_file, width="100%", height="600"))

    # --- Show annotation table for nodes in this neighborhood ---
    with out_table:
        sub_df = df[df["node"].isin(sub_nodes)].copy()
        sub_df["__order"] = (sub_df["node"] != gene_id).astype(int)
        sub_df = sub_df.sort_values(["__order", "node"]).drop(columns="__order")
        display(sub_df)


def on_update_clicked(_):
    show_neighborhood_pyvis(
        gene_dropdown.value,
        max_neighbors=max_neighbors_slider.value,
    )


update_button.on_click(on_update_clicked)

ui = VBox(
    [
        HBox([gene_dropdown, max_neighbors_slider, update_button]),
        Label("Interactive neighborhood (pyvis):"),
        out_plot,
        Label("Annotations for neighborhood nodes:"),
        out_table,
    ]
)

display(ui)

In [25]:
from IPython.display import display, IFrame
from ipywidgets import (
    Dropdown, Button, HBox, VBox, Output, Label, IntSlider, Checkbox
)
from pyvis.network import Network

# Pre-index annotation DataFrame for quick lookup
annot = df.set_index("node")

# Valid nodes that exist in both df and graph
valid_nodes = sorted(annot.index.intersection(G.nodes()))

gene_dropdown = Dropdown(
    options=valid_nodes,
    description="Gene:",
    layout={"width": "300px"}
)

max_neighbors_slider = IntSlider(
    value=30,
    min=5,
    max=200,
    step=5,
    description="Max neighbors:",
    continuous_update=False,
    layout={"width": "300px"}
)

# --- Checkboxes for annotations ---

cb_disorder = Checkbox(
    description="Fraction disordered",   # disorder_fraction
    value=True
)
cb_stable_complex = Checkbox(
    description="Stable complex",        # in_complex
    value=False
)
cb_essential = Checkbox(
    description="Essentiality",          # essential
    value=False
)
cb_chaperones = Checkbox(
    description="Interacts with chaperones",  # interacting_chaperones
    value=False
)
cb_halflife = Checkbox(
    description="Halflife",              # Villen_halflife_hours
    value=False
)
cb_abundance = Checkbox(
    description="Abundance",             # mean_molecules_per_cell
    value=False
)
cb_degree = Checkbox(
    description="Number of edges",       # computed from G.degree
    value=False
)

# Map checkbox -> (display label, "column" id)
# Use a sentinel "__degree__" for the computed degree annotation
ANNOT_OPTIONS = [
    (cb_disorder,   "f_disordered",          "disorder_fraction"),
    (cb_stable_complex, "Stable complex",    "in_complex"),
    (cb_essential,  "Essentiality",          "essential"),
    (cb_chaperones, "Interacts with chaperones", "interacting_chaperones"),
    (cb_halflife,   "Halflife",              "Villen_halflife_hours"),
    (cb_abundance,  "Abundance",             "mean_molecules_per_cell"),
    (cb_degree,     "Number of edges",       "__degree__"),   # NEW
]

update_button = Button(
    description="Show neighbors (pyvis)",
    tooltip="Display the selected node and its immediate neighbors (interactive)",
    button_style=""
)

out_plot = Output()
out_table = Output()


def _format_chaperone_value(val):
    """Pretty-print interacting_chaperones column."""
    if isinstance(val, (list, tuple)):
        if len(val) == 0:
            return "None"
        return ", ".join(map(str, val))
    if isinstance(val, str):
        stripped = val.strip()
        if stripped == "[]" or stripped == "":
            return "None"
        return stripped
    if pd.isna(val):
        return "None"
    return str(val)


def show_neighborhood_pyvis(gene_id: str, max_neighbors: int | None = None):
    """Build the 1-hop neighborhood of gene_id and display it with pyvis."""
    out_plot.clear_output(wait=True)
    out_table.clear_output(wait=True)

    if gene_id not in G:
        with out_plot:
            print(f"{gene_id} is not present in the NetworkX graph.")
        return

    # --- Build 1-hop neighborhood ---
    neighbors = list(G.neighbors(gene_id))

    if max_neighbors is not None and len(neighbors) > max_neighbors:
        neighbors = sorted(neighbors, key=lambda n: G.degree(n), reverse=True)[:max_neighbors]

    sub_nodes = [gene_id] + neighbors
    subG = G.subgraph(sub_nodes).copy()

    # --- Create pyvis network with compact layout ---
    net = Network(
        notebook=True,
        height="600px",
        width="600px",
        bgcolor="white",
        font_color="black",
    )

    net.barnes_hut(
        gravity=-2000,
        central_gravity=0.7,
        spring_length=70,
        spring_strength=0.04,
        damping=0.85,
        overlap=0.1,
    )

    for n in subG.nodes():
        label = n
        tooltip_lines = [str(n)]  # always start with node name

        row = annot.loc[n] if n in annot.index else None

        # Add any selected annotations
        for cb, disp_label, col in ANNOT_OPTIONS:
            if not cb.value:
                continue

            # Special case: degree (number of edges) from graph G
            if col == "__degree__":
                val_str = str(G.degree[n])  # degree in full interactome
                tooltip_lines.append(f"{disp_label}: {val_str}")
                continue

            # For everything else we need row + column
            if row is None or col not in row.index:
                continue

            val = row[col]

            if col == "interacting_chaperones":
                val_str = _format_chaperone_value(val)
            elif col == "disorder_fraction":
                try:
                    val_str = f"{float(val):.3f}"
                except Exception:
                    val_str = str(val)
            else:
                val_str = "None" if pd.isna(val) else str(val)

            tooltip_lines.append(f"{disp_label}: {val_str}")

        # Plain-text tooltip (safe for vis-network)
        title = "\n".join(tooltip_lines)

        # Color & size
        if n == gene_id:
            color = "#0077BB"   # selected node
            size = 25
        else:
            color = "lightgray"
            size = 15

        net.add_node(
            n,
            label=label,
            title=title,
            color=color,
            size=size,
        )

    # Add edges
    for u, v in subG.edges():
        net.add_edge(u, v)

    html_file = "ppi_neighborhood.html"
    net.show(html_file)

    with out_plot:
        display(IFrame(src=html_file, width="100%", height="600"))

    # --- Show annotation table for nodes in this neighborhood ---
    with out_table:
        sub_df = df[df["node"].isin(sub_nodes)].copy()
        sub_df["__order"] = (sub_df["node"] != gene_id).astype(int)
        sub_df = sub_df.sort_values(["__order", "node"]).drop(columns="__order")
        display(sub_df)


def on_update_clicked(_):
    show_neighborhood_pyvis(
        gene_dropdown.value,
        max_neighbors=max_neighbors_slider.value,
    )


update_button.on_click(on_update_clicked)

ui = VBox(
    [
        HBox([gene_dropdown, max_neighbors_slider, update_button]),
        HBox([
            VBox([cb_disorder, cb_stable_complex, cb_essential]),
            VBox([cb_chaperones, cb_halflife, cb_abundance, cb_degree]),
        ]),
        Label("Interactive neighborhood (pyvis):"),
        out_plot,
        Label("Annotations for neighborhood nodes:"),
        out_table,
    ]
)

display(ui)